# スライスではなく catch-all アンパックを使う

アンパックの基本的な制限として、前もってアンパックするシーケンスの長さが必要です。

例えば、車のディーラーが扱っている車の使用年数のリストがあったとします。リストの先頭の2 つをアンパックしようとしたら、実行時にエラーが生じました。

In [1]:
car_ages = [0 , 9, 4, 8, 7, 20, 19, 1, 6, 15]
car_ages_descending = sorted(car_ages, reverse=True)
oldest, second_oldest = car_ages_descending

ValueError: too many values to unpack (expected 2)

Python にはこの状況をより上手に扱えるよう、アスタリスク付きの引数による catch-all アンパックがあります。
この構文では、アンパック代入において、他のアンパックパターンに合致しない残り全部を受け取ることができます。

In [2]:
oldest, second_oldest, *others = car_ages_descending
print(oldest)
print(second_oldest)
print(others)

20
19
[15, 9, 8, 7, 6, 4, 1, 0]


このコードは短くて読みやすく、行間で同期しなければならない境界値エラーが生じる脆弱性がなくなりました。

アスタリスク付きの引数はどこの位置にも欠けるので、スライスでいつでも catch-call アンパックの恩恵を受けることができます。

In [3]:
oldest, *others, youngest = car_ages_descending
print(oldest)
print(youngest)
print(others)

20
0
[19, 15, 9, 8, 7, 6, 4, 1]


In [4]:
*oldest, others, youngest = car_ages_descending
print(oldest)
print(youngest)
print(others)

[20, 19, 15, 9, 8, 7, 6, 4]
0
1


アスタリスク付きの式を含むアンパック代入では、少なくとも1つの指定部分が必要です。

そうでないと、SyntaxError になります。

In [5]:
*others = car_ages_descending  # SyntaxError

SyntaxError: starred assignment target must be in a list or tuple (1068660537.py, line 1)

1つのアンパックパターンの中に複数のアスタリスク付きの式を指定することもできません。

In [6]:
first, *middle, *second_middle, last = [1, 2, 3, 4, 5]  # SyntaxError

SyntaxError: multiple starred expressions in assignment (1208910528.py, line 1)

アスタリスク付きの式はどの場合にも list インスタンスになります。

アンパックされるシーケンスで要素が残っていない場合は、catch-all アンパックの結果は空リストになります。

これは少なくとも N 要素あると前もってわかっているシーケンス処理では特に役立ちます。

In [7]:
short_list = [1, 2]
first, second, *rest = short_list
print(first, second, rest)  # rest is []

1 2 []


アスタリスク付きの式を付け加えれば、イテレータをアンパックした値はより明確になります。例えば、次のように、今秋ディーラーから受け取ったすべての車の注文の CSV ファイルの行を yield するジェネレータがあるとします。

In [9]:
def generate_csv():
    # Header row
    yield ('Date', 'Make', 'Model', 'Year', 'Price')
    
    # 以下はダミーの注文データ
    yield ('2025-11-01', 'Toyota', 'Corolla', '2025', '$20,000')
    yield ('2025-11-02', 'Honda', 'Civic', '2024', '$22,500')
    yield ('2025-11-02', 'Ford', 'Mustang', '2025', '$45,000')
    yield ('2025-11-03', 'Nissan', 'Leaf', '2023', '$28,000')
    yield ('2025-11-03', 'Chevrolet', 'Camaro', '2025', '$42,500')
    yield ('2025-11-03', 'BMW', '330i', '2024', '$48,000')
    yield ('2025-11-04', 'Audi', 'A4', '2024', '$47,000')
    yield ('2025-11-04', 'Mercedes', 'C300', '2024', '$51,000')
    yield ('2025-11-05', 'Tesla', 'Model 3', '2025', '$39,000')

インデックスとスライスを使ってこのジェネレータの結果を処理してもいいのですが、行が複数になり、見た目がすっきりしません。

In [10]:
all_csv_rows = list(generate_csv())
header, *rows = all_csv_rows

print('CSV Header:', header)
print('Row Count:', len(rows))

CSV Header: ('Date', 'Make', 'Model', 'Year', 'Price')
Row Count: 9


アスタリスク付きの式でアンパックすると、第1行のヘッダをイテレータの残りの内容と別々に処理することが簡単になり、より明確になります。

In [11]:
it = generate_csv()
header, *rows = it
print('CSV Header:', header)
print('Row Count:', len(rows))

CSV Header: ('Date', 'Make', 'Model', 'Year', 'Price')
Row Count: 9


しかし、アスタリスク付きの式が常にリストを返すので、イテレータのアンパックにはコンピュータの全メモリを使いつくしてプログラムがクラッシュする危険のあることを忘れないでください。

結果のデータがメモリに収まるという確信が持てる場合にのみ（項目31）イテレータに cathc-all アンパックを用いるべきです。

## 覚えておくこと

- 代入のアンパックでアスタリスク付きの式を使い、アンパックするパターンの残りをまとめてリストにできる
- アスタリスク付きの式はどこにでも書くことができ、常に受け取る値のリストになる。
- リストを重複のないように分割する場合、catch-all アンパックは、スライスやインデックスを使うよりもエラーを起こす危険が少ない。

## 補足

「イテレータはデータを持っていない」という説明が謎に感じるところを、しっかり整理します。

### まず、イテレータとは？

イテレータは「次の要素を生成する方法」だけ持つものです。

> イテレータは データそのものを全部メモリに持っていない
→ 次の値を要求されたときに「計算して返す」

In [ ]:
def generate_csv():
    yield ('Date', 'Make', 'Model')
    yield ('2025-11-01', 'Toyota', 'Corolla')
    # ...

この generate_csv() の内部には
「どんな順番で次のデータを 作るか」というコードがあるだけで、

- データを全部格納しているわけではない

つまり、まだ出していないデータはメモリ上に存在しない！

### 従来のリストとの違い

| 種類                 | メモリに持つもの                         |
| ------------------ | -------------------------------- |
| **リスト**            | 全要素を保存する（200 行 → 200 行分全部メモリ使用）  |
| **イテレータ / ジェネレータ** | 次に返す 1 行分だけ（200 行 → 1 行分だけメモリ使用） |

→巨大なデータに強い（ストリーミング処理）

### catch-all アンパック で問題が起きる理由

header, *rows = generate_csv()

この *rows が曲者。

rows に 残り全部 入れようとする
→ リストに展開される
→ 結局、全行メモリに読み込む

つまり：

>元がイテレータでも、アンパックしてリスト化すると
イテレータの利点は消える！

### では「データはどこにあるの？」

| 状況                         | データはどこにある？          |
| -------------------------- | ------------------- |
| ファイルやネットワークを逐次読み込み         | ディスク or 通信先から1件ずつ取得 |
| 計算で生成するジェネレータ              | CPU が必要になると都度作る     |
| list(iterator) や *rows で展開 | RAM中（全件メモリ展開）       |


### 結論

イテレータ自体は全データを持たない
　→ 最小限の状態だけ保持

ただし、リスト化した瞬間に全量メモリを消費する

- list(iterator)
- header, *rows = iterator（キャッチオール）
- スライス等

### 覚えておくべき区別

| 書き方                      | メモリ使用          |
| ------------------------ | -------------- |
| `for row in csv_reader:` | ほぼ一定（大規模データOK） |
| `*rows = csv_reader`     | データ全部読み込む💥    |

> 大量データにはイテレータを活かすべし
少量データならアンパック便利